In [ ]:
!pip uninstall -y jax jaxlib

import tensorflow as tf
import pandas as pd
from tensorflow import keras

!pip install tensorflow-datasets

import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Load datasets
train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

train_df = pd.read_csv(
    train_file_path,
    sep='\t',
    header=None,
    names=['label', 'message']
)

test_df = pd.read_csv(
    test_file_path,
    sep='\t',
    header=None,
    names=['label', 'message']
)

# Convert labels
train_labels = train_df['label'].map({'ham': 0, 'spam': 1})
test_labels = test_df['label'].map({'ham': 0, 'spam': 1})

# Tokenizer
vocab_size = 10000
max_length = 100
padding_type = 'post'
trunc_type = 'post'
oov_tok = "<OOV>"

tokenizer = Tokenizer(
    num_words=vocab_size,
    oov_token=oov_tok
)

tokenizer.fit_on_texts(train_df['message'])

# Convert text into sequences
train_sequences = tokenizer.texts_to_sequences(train_df['message'])
test_sequences = tokenizer.texts_to_sequences(test_df['message'])

# Pad sequences
train_padded = pad_sequences(
    train_sequences,
    maxlen=max_length,
    padding=padding_type,
    truncating=trunc_type
)

test_padded = pad_sequences(
    test_sequences,
    maxlen=max_length,
    padding=padding_type,
    truncating=trunc_type
)

# Build model
model = Sequential([
    Embedding(vocab_size, 32, input_length=max_length),
    LSTM(32),
    Dense(24, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compile model
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# Train model
model.fit(
    train_padded,
    train_labels,
    epochs=10,
    validation_data=(test_padded, test_labels),
    verbose=2
)

# Prediction function
def predict_message(pred_text):

    sequence = tokenizer.texts_to_sequences([pred_text])

    padded = pad_sequences(
        sequence,
        maxlen=max_length,
        padding=padding_type,
        truncating=trunc_type
    )

    prediction = model.predict(padded)[0][0]

    label = "spam" if prediction > 0.5 else "ham"

    return [float(prediction), label]

In [ ]:
# function to predict messages based on model
# should return [probability, label]

def predict_message(pred_text):

    sequence = tokenizer.texts_to_sequences([pred_text])

    padded = pad_sequences(
        sequence,
        maxlen=max_length,
        padding='post'
    )

    prediction = model.predict(padded)[0][0]

    label = "spam" if prediction > 0.5 else "ham"

    return [float(prediction), label]


pred_text = "how are you doing today?"

prediction = predict_message(pred_text)

print(prediction)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
